## 饿了么解读

[原始数据获取地址](https://tianchi.aliyun.com/dataset/123953)


示例行（原始）
```cs
0,c9f05d1754eb14421607e37e733db9824b419c2498b0b2f9c283c66a73f5d18a,2,1,59.195,1,60,4,236.78,063167b7d5120c432a6878b8bf85eac0a1d13b080572d0114e587513e177dae2,7655b3f9d9bfc5fc0b163957e5feec15eb45d160015718634b727f5699e68a56,1,5265,B0FFGBBNVA,wtw1kb,wtw1kbue7hy1,1085035,1001,16730,1,1,1,f141b69ef532...;...,9509498a17...;...,1071;1059;1106;1059,0;0;13662;0,726906;726906;726906;726906,3.8;2.8;5.8;0.0,B00157GJNB;...,wtw1hn;...,2285;2301;2397;2441,12;12;12;12,lunch;lunch;lunch;lunch,0;0;0;0,1646628521,12,lunch,0,wtw1kcxch4e5
```
逐字段映射与解读

`label`: `0` — 样本标签（0/1），此行为负样本（未转化/未点击/未下单，取决于任务定义）。\
`user_id`: `c9f05d1...` — 用户ID（已哈希/编码），用来关联用户历史与画像。\
`gender`: `2` — 性别编码（类别型，需与数据说明映射到具体含义）。\
`visit_city`: `1` — 用户所在/访问城市的ID（离散编码）。\
`avg_price`: `59.195` — 用户平均消费（元），反映用户典型客单价。\
`is_supervip`: `1` — 是否超级会员（`1`=是）。\
`ctr_30`: `60` — 最近30天点击率/相关交互指标（数值编码，需确认是百分比/千分比/原始计数）。\
`ord_30`: `4` — 最近30天下单次数（`4` 次）。\
`total_amt_30`: `236.78` — 最近30天消费总额（元）。


#### 候选商品/店铺属性：

`shop_id`: `063167b7...` — 候选店铺ID（哈希）。\
`item_id`: `7655b3f9...` — 候选商品/菜品ID（哈希）。\
`city_id`: `1` — 店铺所在城市ID（与 `visit_city` 同编码体系）。\
`district_id`: `5265` — 店铺所在区/商圈ID。\
`shop_aoi_id`: `B0FFGBBNVA` — 店铺 AOI/兴趣点 ID（服务区/小区级别）。\
`shop_geohash_6` / `shop_geohash_12`: `wtw1kb` / `wtw1kbue7hy1` — 店铺位置 geohash（用于距离/空间特征）。\
`brand_id`: `1085035` — 品牌ID。\
`category_1_id`: `1001` — 一级类目ID。\
`merge_standard_food_id`: `16730` — 标准化菜品ID（用于菜品级相似度）。\
`rank_7` / `rank_30` / `rank_90`: `1`, `1`, `1` — 商品/店铺最近7/30/90天的排名或热度指标（数值越小/大含义依赖定义）。


#### 用户行为序列（分号 ; 分隔，序列长度一致，对应最近若干次交互）：

`shop_id_list`: `f141b69e...;f141...;...` — 历史交互过的店铺序列（重复表示多次访问同店）。\
`item_id_list`: `9509498a...;bdfc...;1049...;3132...` — 历史交互商品ID序列。\
`category_1_id_list`: `1071;1059;1106;1059` — 历史交互的一级类目序列。\
`merge_standard_food_id_list`: `0;0;13662;0` — 历史交互的标准化菜品ID序列（0 可能表示未命中标准化项）。\
`brand_id_list`: `726906;726906;726906;726906` — 历史交互品牌序列（用户多次与同一品牌交互）。\
`price_list`: `3.8;2.8;5.8;0.0` — 每次交互时商品/配送相关价格序列（元）。\
`shop_aoi_id_list`: `B00157GJNB;...` — 历史触达的 AOI 列表（常用于位置信息聚合）。\
`shop_geohash6_list`: `wtw1hn;wtw1hn;wtw1hn;wtw1hn` — 历史交互时店铺的 geohash6 序列。\
`timediff_list`: `2285;2301;2397;2441` — 每次历史交互距当前时间的时间差（单位通常为秒）。\
`hours_list`: `12;12;12;12` — 历史交互发生的小时时段（均为 12 点）。\
`time_type_list`: `lunch;lunch;lunch;lunch` — 历史交互对应餐别（如午餐）。\
`weekdays_list`: `0;0;0;0` — 历史交互对应星期编码（0 代表某一天，需按数据字典映射）。


#### 当前时空上下文：

`times`: `1646628521` — 当前请求时间戳（Unix 秒）。\
`hours`: `12` — 当前小时（12 点，午餐时段）。\
`time_type`: `lunch` — 当前时间段类别（午餐）。\
`weekdays`: `0` — 当前星期编码（需映射解释）。\
`geohash12`: `wtw1kcxch4e5` — 用户当前位置 geohash12（用于计算与店铺距离/可配送性）。

## DLRM 用数据预处理步骤

使用 `preprocess.py` 将 Eleme 原始 CSV 转换为 Criteo 风格的 TSV，并生成 schema 与 hashmap。
主要输出（默认写入 `output/preproc/`）：
- 转换后的数据: `<output>.tsv`
- Schema: `<output>.tsv.schema.json`（JSON，顶层键 `columns`）
- Hash 映射: `<output>.tsv.hashmap.json`（每个稀疏字段的 token->hash 映射）
- 可选样例: `<output>.tsv.head.tsv`（表头 + 前 N 行）

常用命令示例（在工作区根目录运行）：
```bash
python preprocess.py --fields eleme_fields.csv --input eleme_sample.csv --output eleme_criteo_checked.tsv --chunk_size 200 --workers 4 --output-dir output/preproc --head-sample 20
```

在 notebook 中也可直接运行：
```python
%run preprocess.py --fields eleme_fields.csv --input eleme_sample.csv --output eleme_criteo_checked.tsv --chunk_size 200 --workers 1 --output-dir output/preproc --head-sample 20
```

参数说明：
- `--fields`: 字段说明 CSV（示例: `eleme_fields.csv`）
- `--input`: 原始 CSV 输入文件
- `--output`: 输出文件基名（写入 `--output-dir`）
- `--chunk_size`: 每个并行任务处理的行数（较小可降低内存）
- `--workers`: 并行进程数（0 表示使用全部 CPU）
- `--output-dir`: 输出目录（默认 `output/preproc`）
- `--head-sample N`: 可选，写出带表头的前 N 行样例；不带值则默认为 20；传 0 禁用

行为细节注意：数值型输出均取整数（对金额类字段乘以 100 后取整），序列字段会展开统计列（如 `*_len`, `*_uniq`, `*_sum`, `*_mean`, `*_std`, `*_min`, `*_max`），类别序列会输出 top-k 的 token_hash 与 count，token_hash 使用 md5 的前 8 个 hex 字符。

如需调整 head 样例行数，可在命令中将 `--head-sample 20` 改为任意正整数（或省略以使用默认 20）。

## 预处理脚本

In [1]:
%pip install -r requirements.txt
%run scripts/preprocess.py --fields eleme_fields.csv --input eleme_sample.csv --output eleme_criteo_checked.tsv --chunk_size 200 --workers 1 --output-dir output/preproc --head-sample 20


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


processing: 100%|██████████| 20/20 [00:00<00:00, 83.78it/s]

schema written: output/preproc\eleme_criteo_checked.tsv.schema.json
hash mapping written: output/preproc\eleme_criteo_checked.tsv.hashmap.json
head sample written: output/preproc\eleme_criteo_checked.tsv.head.tsv (20 rows)


### 稀疏特征取值数量（Vocabulary）计数

In [8]:
%run count_unique_sparse.py --input ../DATASET/Eleme/20220307_01.csv  --feature-names label user_id gender visit_city avg_price is_supervip ctr_30 ord_30 total_amt_30 shop_id item_id city_id district_id shop_aoi_id shop_geohash_6 shop_geohash_12 brand_id category_1_id merge_standard_food_id rank_7 rank_30 rank_90 shop_id_list item_id_list category_1_id_list merge_standard_food_id_list brand_id_list price_list shop_aoi_id_list shop_geohash6_list timediff_list hours_list time_type_list weekdays_list times hours time_type weekdays geohash12 --sparse-cols label user_id gender visit_city is_supervip shop_id item_id city_id district_id shop_aoi_id shop_geohash_6 shop_geohash_12 brand_id category_1_id merge_standard_food_id shop_id_list item_id_list category_1_id_list merge_standard_food_id_list brand_id_list price_list shop_aoi_id_list shop_geohash6_list timediff_list hours_list time_type_list weekdays_list time_type weekdays geohash12 --chunksize 200000  --topk 20 --output eleme_unique_counts.json

Saved results to eleme_unique_counts.json
